# SHROOM Final Colab Notebook

**Final implementation for submission/review**  
This notebook reproduces the final model-agnostic experiments for hallucination detection in SHROOM.

The heavy steps have already been computed and are **not rerun here**:
- NLI feature extraction with DeBERTa
- Semantic similarity embedding extraction
- Synthetic data generation with Gemini/API

Instead, this Colab notebook loads the precomputed feature file:

```text
nli_similarity_features_dataset.csv
```

Then it reruns the reproducibility/evaluation experiments:

| Experiment | Description |
|---|---|
| E1 | NLI-only Random Forest meta-regressor |
| E2 | Semantic similarity-only Random Forest meta-regressor |
| E3 | NLI + semantic similarity Random Forest meta-regressor |

The final selected model is **E3: NLI + Semantic Similarity**, because it achieved the best overall performance in the ablation study.


## 1. Setup

Run this cell first. It installs/loads only lightweight packages. No GPU, API key, or HuggingFace model download is required.


In [ ]:
%pip -q install pandas numpy scikit-learn scipy

import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLD = 0.5

np.random.seed(RANDOM_STATE)

print("Configuration")
print("RANDOM_STATE:", RANDOM_STATE)
print("N_SPLITS:", N_SPLITS)
print("THRESHOLD:", THRESHOLD)


## 2. Upload / Load the precomputed feature dataset

When prompted, upload:

```text
nli_similarity_features_dataset.csv
```

This file must contain the original labeled model-agnostic evaluation data together with the precomputed NLI and semantic similarity features.


In [ ]:
DATA_PATH = "nli_similarity_features_dataset.csv"

if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        print(f"Please upload {DATA_PATH}")
        uploaded = files.upload()

        if DATA_PATH not in uploaded:
            csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
            if len(csv_files) == 1:
                DATA_PATH = csv_files[0]
                print(f"Using uploaded CSV file: {DATA_PATH}")
            else:
                raise FileNotFoundError(
                    f"Expected {DATA_PATH}. Uploaded files: {list(uploaded.keys())}"
                )
    except ModuleNotFoundError:
        raise FileNotFoundError(
            f"{DATA_PATH} was not found. Place it in the same folder as this notebook or upload it in Colab."
        )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


## 3. Normalize columns and validate the dataset

This cell makes the notebook robust to small naming differences, for example `p(Hallucination)` vs `p_hallucination`.


In [ ]:
# Normalize probability column.
if "p_hallucination" not in df.columns:
    if "p(Hallucination)" in df.columns:
        df["p_hallucination"] = df["p(Hallucination)"]
    else:
        raise ValueError("Missing probability column: expected p_hallucination or p(Hallucination).")

# Normalize binary label column.
if "label_bin" not in df.columns:
    if "label" in df.columns:
        df["label_bin"] = df["label"].map({
            "Hallucination": 1,
            "Not Hallucination": 0,
        })
    else:
        raise ValueError("Missing label column: expected label_bin or label.")

# Basic cleanup.
df["task"] = df["task"].astype(str).str.strip()
df["label_bin"] = df["label_bin"].astype(int)
df["p_hallucination"] = df["p_hallucination"].astype(float)

print("Task distribution:")
display(df["task"].value_counts().rename("count").to_frame())

print("Label distribution:")
display(df["label_bin"].value_counts().sort_index().rename(index={0: "Not Hallucination", 1: "Hallucination"}).rename("count").to_frame())

print("p_hallucination distribution:")
display(df["p_hallucination"].value_counts().sort_index().rename("count").to_frame())


## 4. Feature groups

The final pipeline uses two groups of features:

1. **NLI features**: entailment, neutral, contradiction for `src ↔ hyp` and `tgt ↔ hyp`.
2. **Semantic similarity features**: cosine similarity features between `src`, `tgt`, and `hyp`.


In [ ]:
nli_feature_cols = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con",
]

similarity_feature_cols = [
    "sim_src_hyp",
    "sim_tgt_hyp",
    "sim_src_tgt",
    "sim_max",
    "sim_min",
    "sim_mean",
    "sim_gap_tgt_src",
    "sim_hyp_distance_from_truth",
]

nli_similarity_feature_cols = nli_feature_cols + similarity_feature_cols

required_cols = ["task", "label_bin", "p_hallucination"] + nli_similarity_feature_cols
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(
        "The feature dataset is missing required columns:\n" + "\n".join(missing_cols)
    )

# Keep only rows with complete required data.
before_rows = len(df)
df = df.dropna(subset=required_cols).reset_index(drop=True)
after_rows = len(df)

print(f"Rows before dropna: {before_rows}")
print(f"Rows after dropna:  {after_rows}")
print("NLI features:", len(nli_feature_cols))
print("Similarity features:", len(similarity_feature_cols))
print("Combined features:", len(nli_similarity_feature_cols))


## 5. Evaluation helper

Each experiment uses a `RandomForestRegressor` as a meta-learner.  
The model predicts `p_hallucination`, and the binary label is obtained with threshold `0.5`.


In [ ]:
def safe_spearman(y_true_prob, y_pred_prob):
    corr = spearmanr(y_true_prob, y_pred_prob).correlation
    if pd.isna(corr):
        return 0.0
    return float(corr)


def evaluate_predictions(y_true_bin, y_true_prob, y_pred_prob, threshold=THRESHOLD):
    y_true_bin = np.asarray(y_true_bin)
    y_true_prob = np.asarray(y_true_prob)
    y_pred_prob = np.asarray(y_pred_prob)

    y_pred_bin = (y_pred_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true_bin, y_pred_bin),
        "spearman": safe_spearman(y_true_prob, y_pred_prob),
    }


def run_rf_cv(dataframe, feature_cols, experiment_id, experiment_name):
    X = dataframe[feature_cols].values
    y_reg = dataframe["p_hallucination"].values
    y_bin = dataframe["label_bin"].values
    tasks = dataframe["task"].values

    stratify_key = dataframe["task"].astype(str) + "_" + dataframe["label_bin"].astype(str)

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    fold_results = []
    all_val_preds = []
    all_val_true_reg = []
    all_val_true_bin = []
    all_val_tasks = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, stratify_key), start=1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train_reg = y_reg[train_idx]
        y_val_reg = y_reg[val_idx]
        y_val_bin = y_bin[val_idx]
        val_tasks = tasks[val_idx]

        model = RandomForestRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X_train, y_train_reg)

        val_pred = model.predict(X_val)
        val_pred = np.clip(val_pred, 0.0, 1.0)

        metrics = evaluate_predictions(
            y_true_bin=y_val_bin,
            y_true_prob=y_val_reg,
            y_pred_prob=val_pred,
            threshold=THRESHOLD,
        )

        fold_results.append({
            "experiment_id": experiment_id,
            "experiment_name": experiment_name,
            "fold": fold,
            "accuracy": metrics["accuracy"],
            "spearman": metrics["spearman"],
        })

        all_val_preds.extend(val_pred)
        all_val_true_reg.extend(y_val_reg)
        all_val_true_bin.extend(y_val_bin)
        all_val_tasks.extend(val_tasks)

        print(
            f"{experiment_id} | Fold {fold}: "
            f"Accuracy={metrics['accuracy']:.4f}, "
            f"Spearman={metrics['spearman']:.4f}"
        )

    cv_results_df = pd.DataFrame(fold_results)

    oof_df = pd.DataFrame({
        "experiment_id": experiment_id,
        "experiment_name": experiment_name,
        "task": all_val_tasks,
        "y_true_reg": all_val_true_reg,
        "y_true_bin": all_val_true_bin,
        "y_pred_reg": all_val_preds,
    })
    oof_df["y_pred_bin"] = (oof_df["y_pred_reg"] >= THRESHOLD).astype(int)

    overall_metrics = evaluate_predictions(
        y_true_bin=oof_df["y_true_bin"],
        y_true_prob=oof_df["y_true_reg"],
        y_pred_prob=oof_df["y_pred_reg"],
        threshold=THRESHOLD,
    )

    summary = {
        "experiment_id": experiment_id,
        "experiment_name": experiment_name,
        "n_features": len(feature_cols),
        "mean_cv_accuracy": cv_results_df["accuracy"].mean(),
        "mean_cv_spearman": cv_results_df["spearman"].mean(),
        "overall_oof_accuracy": overall_metrics["accuracy"],
        "overall_oof_spearman": overall_metrics["spearman"],
    }

    per_task_rows = []
    for task_name in sorted(oof_df["task"].unique()):
        task_df = oof_df[oof_df["task"] == task_name]
        task_metrics = evaluate_predictions(
            y_true_bin=task_df["y_true_bin"],
            y_true_prob=task_df["y_true_reg"],
            y_pred_prob=task_df["y_pred_reg"],
            threshold=THRESHOLD,
        )
        per_task_rows.append({
            "experiment_id": experiment_id,
            "experiment_name": experiment_name,
            "task": task_name,
            "accuracy": task_metrics["accuracy"],
            "spearman": task_metrics["spearman"],
            "n_samples": len(task_df),
        })

    per_task_df = pd.DataFrame(per_task_rows)

    return summary, cv_results_df, oof_df, per_task_df


## 6. Run E1, E2 and E3


In [ ]:
experiments = [
    ("E1", "NLI only", nli_feature_cols),
    ("E2", "Semantic similarity only", similarity_feature_cols),
    ("E3", "NLI + Semantic similarity", nli_similarity_feature_cols),
]

summary_rows = []
cv_results = {}
oof_predictions = {}
per_task_results = {}

for experiment_id, experiment_name, feature_cols in experiments:
    print("\n" + "=" * 80)
    print(f"Running {experiment_id}: {experiment_name}")
    print("=" * 80)

    summary, cv_df, oof_df, per_task_df = run_rf_cv(
        dataframe=df,
        feature_cols=feature_cols,
        experiment_id=experiment_id,
        experiment_name=experiment_name,
    )

    summary_rows.append(summary)
    cv_results[experiment_id] = cv_df
    oof_predictions[experiment_id] = oof_df
    per_task_results[experiment_id] = per_task_df

summary_df = pd.DataFrame(summary_rows)
summary_df


## 7. Final results table

Use the `overall_oof_accuracy` and `overall_oof_spearman` columns as the main reported results.


In [ ]:
final_results_df = summary_df[[
    "experiment_id",
    "experiment_name",
    "n_features",
    "mean_cv_accuracy",
    "mean_cv_spearman",
    "overall_oof_accuracy",
    "overall_oof_spearman",
]].copy()

# Round only for display. The saved CSV keeps full precision.
display(final_results_df.round(4))

best_row = final_results_df.sort_values(
    by=["overall_oof_accuracy", "overall_oof_spearman"],
    ascending=False,
).iloc[0]

print("Best experiment:")
print(best_row.to_string())


## 8. Per-task analysis for the final model E3


In [ ]:
e3_per_task_df = per_task_results["E3"].copy()
display(e3_per_task_df.round(4))


## 9. Feature importance for the final E3 model

This trains the final Random Forest meta-learner on the whole available dataset only to inspect feature importances.


In [ ]:
final_model = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_model.fit(df[nli_similarity_feature_cols].values, df["p_hallucination"].values)

importance_df = pd.DataFrame({
    "feature": nli_similarity_feature_cols,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(importance_df.round(4))


## 10. E4 synthetic-data ablation note

E4 was tested as an ablation with synthetic data, but it is **not rerun by default** in this Colab notebook because full reproduction requires the synthetic training features as a separate precomputed file.

In the current implementation, E4 performed worse than E3. Therefore, the final selected system remains **E3: NLI + Semantic Similarity**.

Reported E4 result from the implementation logs:

| Experiment | Accuracy | Spearman ρ | Interpretation |
|---|---:|---:|---|
| E4: NLI + Similarity + Synthetic Data | 0.7113 | 0.6318 | Synthetic data did not improve the final model in the current setup |

This is kept as an ablation finding and can be discussed as evidence that weak supervision requires stricter filtering and/or better label quality.


## 11. Save results

This cell saves the reproduced evaluation results as CSV files.


In [ ]:
OUTPUT_DIR = Path("shroom_final_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_results_df.to_csv(OUTPUT_DIR / "final_results_summary.csv", index=False)
importance_df.to_csv(OUTPUT_DIR / "e3_feature_importance.csv", index=False)

for experiment_id in cv_results:
    cv_results[experiment_id].to_csv(OUTPUT_DIR / f"{experiment_id.lower()}_cv_results.csv", index=False)
    oof_predictions[experiment_id].to_csv(OUTPUT_DIR / f"{experiment_id.lower()}_oof_predictions.csv", index=False)
    per_task_results[experiment_id].to_csv(OUTPUT_DIR / f"{experiment_id.lower()}_per_task_results.csv", index=False)

print("Saved results in:", OUTPUT_DIR.resolve())
print("Files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path.name)


## 12. Final conclusion

Based on the ablation study, the final selected implementation is:

> **E3 — Random Forest meta-regressor using NLI + semantic similarity features**

This model is preferred because it combines semantic reasoning signals from NLI with semantic proximity signals from sentence similarity, and it achieved the best overall performance among the tested configurations.
